# Expériences complémentaires : test de raccourci, validation croisée, comparaison d'architectures

**Mémoire de Master, ALAWO Adeshina Néhémiah (ESGIS)**

Ce notebook prolonge `memoire_cnn_colab.ipynb` avec **exactement le même protocole** (prétraitement,
partition par patient, deux phases d'entraînement, seuil de Youden, bootstrap). Il lance, dans l'ordre :

| Priorité | Expérience | Question posée |
|---|---|---|
| 1 | Pneumonie : images complètes / bords masqués / **bords seuls** (DenseNet-121) | Le modèle s'appuie-t-il sur les bords du cliché plutôt que sur les poumons ? |
| 2 | Tuberculose : Shenzhen → Montgomery et Montgomery → Shenzhen | Le modèle fonctionne-t-il dans un autre hôpital ? |
| 3 | ResNet-50 et EfficientNet-B0 sur les trois tâches | DenseNet-121 était-il le bon choix ? |

## Mode d'emploi (Kaggle)

1. Créer un notebook Kaggle, *File → Import Notebook*, choisir ce fichier.
2. Panneau de droite : *Accelerator → GPU T4*, *Internet → On*.
3. Ajouter les deux jeux de données (*Add Input*) : `paultimothymooney/chest-xray-pneumonia` et
   `kmader/pulmonary-chest-xray-abnormalities`. Sinon le notebook les télécharge lui-même.
4. **Save Version → Save & Run All (Commit)** : l'exécution se fait en arrière-plan (environ 3 h 30),
   on peut fermer le navigateur. À la fin, l'onglet *Output* contient `memoire_complements.zip`.

Chaque expérience est enregistrée dès qu'elle se termine : en cas de coupure, relancer saute celles déjà faites.

In [ ]:
%pip install -q kagglehub tabulate

## 1. Configuration

In [ ]:
import os, re, io, copy, json, time, random, shutil, hashlib, platform, datetime, zipfile
from collections import OrderedDict
from pathlib import Path
from concurrent.futures import ThreadPoolExecutor

import numpy as np
import pandas as pd
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
import torchvision
from torchvision import transforms
from PIL import Image
import matplotlib
import matplotlib.ticker
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, roc_curve, confusion_matrix

# Protocole identique à memoire_cnn_colab.ipynb : ne pas modifier sans reporter le changement au Chapitre 2
CONFIG = {
    'seed': 42,
    'taille_cache': 256,
    'taille_entree': 224,
    'batch_size': 32,
    'num_workers': 2,
    'partition': (0.70, 0.15, 0.15),
    'dropout': 0.5,
    'weight_decay': 1e-5,
    'plateau': {'factor': 0.5, 'patience': 5},
    'phase1': {'lr': 1e-4, 'max_epochs': 10, 'patience': 10},
    'phase2': {'lr': 1e-5, 'max_epochs': 40, 'patience': 10},
    'poids_imagenet': True,
    'bootstrap': 1000,
    # Nouveau : largeur de la bande masquée sur chaque bord, en fraction du côté de l'image
    'bande_masque': 0.15,
}

# (tâche, architecture, masque) dans l'ordre de priorité
EXPERIENCES = [
    ('pneumonie', 'densenet121', 'aucun'),
    ('pneumonie', 'densenet121', 'bords_masques'),
    ('pneumonie', 'densenet121', 'centre_masque'),
    ('tb_shenzhen_vers_montgomery', 'densenet121', 'aucun'),
    ('tb_montgomery_vers_shenzhen', 'densenet121', 'aucun'),
    ('tuberculose', 'densenet121', 'aucun'),
    ('bacterien_viral', 'densenet121', 'aucun'),
    ('pneumonie', 'resnet50', 'aucun'),
    ('tuberculose', 'resnet50', 'aucun'),
    ('bacterien_viral', 'resnet50', 'aucun'),
    ('pneumonie', 'efficientnet_b0', 'aucun'),
    ('tuberculose', 'efficientnet_b0', 'aucun'),
    ('bacterien_viral', 'efficientnet_b0', 'aucun'),
]
REENTRAINER = False


def nom_experience(tache, archi, masque):
    return f'{tache}__{archi}__{masque}'


def fixer_graine(seed=CONFIG['seed']):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark = False


fixer_graine()
DEVICE = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
GPU = torch.cuda.get_device_name(0) if DEVICE.type == 'cuda' else 'CPU'
if DEVICE.type != 'cuda':
    print("⚠️  Aucun GPU détecté : activer le GPU T4, puis relancer.")

# Les machines Kaggle ont aussi le module google.colab : on teste un fichier propre aux vraies machines Colab
IN_COLAB = os.path.exists('/var/colab/hostname')
if IN_COLAB:
    from google.colab import drive, files as colab_files

if IN_COLAB:
    drive.mount('/content/drive')
    OUT_DIR = Path('/content/drive/MyDrive/memoire_complements')
    CACHE_DIR = Path('/content/cache')
else:
    OUT_DIR = Path('memoire_complements')
    CACHE_DIR = Path('cache')
OUT_DIR.mkdir(parents=True, exist_ok=True)
CACHE_DIR.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({'figure.dpi': 110, 'savefig.dpi': 300, 'font.size': 10,
                     'axes.spines.top': False, 'axes.spines.right': False})
print(f'Matériel : {GPU} | PyTorch {torch.__version__} | torchvision {torchvision.__version__}')
print(f'Résultats : {OUT_DIR.resolve()}')

## 2. Données (identique au premier notebook)

In [ ]:
try:
    from google.colab import userdata
    for cle in ('KAGGLE_USERNAME', 'KAGGLE_KEY', 'KAGGLE_API_TOKEN'):
        try:
            os.environ[cle] = userdata.get(cle)
        except Exception:
            pass
except ImportError:
    pass

import kagglehub
try:
    KERMANY_DIR = Path(kagglehub.dataset_download('paultimothymooney/chest-xray-pneumonia'))
    TB_DIR = Path(kagglehub.dataset_download('kmader/pulmonary-chest-xray-abnormalities'))
except Exception as e:
    raise RuntimeError("Téléchargement Kaggle impossible : vérifier les secrets KAGGLE_* "
                       "(voir le mode d'emploi en haut du notebook).") from e
print(KERMANY_DIR, TB_DIR, sep='\n')

In [ ]:
EXTENSIONS = {'.jpeg', '.jpg', '.png'}


def indexer_kermany(racine):
    lignes, vues = [], set()
    for p in sorted(Path(racine).rglob('*')):
        if p.suffix.lower() not in EXTENSIONS or '__MACOSX' in p.parts:
            continue
        classe = p.parent.name.upper()
        if classe not in ('NORMAL', 'PNEUMONIA'):
            continue
        # L'archive Kaggle contient chest_xray/ en double : on dédoublonne sur (dossier, classe, nom)
        cle = (p.parent.parent.name.lower(), classe, p.name)
        if cle in vues:
            continue
        vues.add(cle)
        if classe == 'NORMAL':
            m = re.search(r'IM-(\d+)', p.name)
            patient, sous_type = f"N-{m.group(1) if m else p.stem}", 'normal'
        else:
            m = re.match(r'(person\d+)_(bacteria|virus)', p.name, re.I)
            patient = f"P-{m.group(1).lower() if m else p.stem}"
            sous_type = m.group(2).lower() if m else 'inconnu'
        lignes.append({'path': str(p), 'source': 'Kermany', 'patient': patient,
                       'classe': classe, 'sous_type': sous_type})
    return pd.DataFrame(lignes)


def indexer_tuberculose(racine):
    lignes, vues = [], set()
    for p in sorted(Path(racine).rglob('*.png')):
        m = re.match(r'^(MCUCXR|CHNCXR)_(\d+)_([01])\.png$', p.name, re.I)
        if not m or 'mask' in str(p.parent).lower() or p.name.upper() in vues:
            continue
        vues.add(p.name.upper())
        lignes.append({'path': str(p),
                       'source': 'Montgomery' if m.group(1).upper() == 'MCUCXR' else 'Shenzhen',
                       'patient': f'{m.group(1).upper()}_{m.group(2)}',
                       'classe': 'TB' if m.group(3) == '1' else 'NORMAL', 'sous_type': ''})
    return pd.DataFrame(lignes)


kermany = indexer_kermany(KERMANY_DIR)
tb = indexer_tuberculose(TB_DIR)
print(f'Kermany : {len(kermany)} images (attendu : 5 856)')
print(kermany.groupby(['classe', 'sous_type']).size().to_string(), '\n')
print(f'Montgomery + Shenzhen : {len(tb)} images (attendu : 800)')
print(tb.groupby(['source', 'classe']).size().to_string())

TACHES = {
    'pneumonie': {
        'titre': 'Pneumonie', 'libelles': ('Normal', 'Pneumonie'), 'pathologie': True,
        'df': kermany.assign(y=(kermany.classe == 'PNEUMONIA').astype(int)),
    },
    'tuberculose': {
        'titre': 'Tuberculose', 'libelles': ('Normal', 'Tuberculose'), 'pathologie': True,
        'df': tb.assign(y=(tb.classe == 'TB').astype(int)),
    },
    'bacterien_viral': {
        'titre': 'Pneumonie bactérienne vs virale', 'libelles': ('Bactérienne', 'Virale'), 'pathologie': False,
        'df': kermany[kermany.sous_type.isin(['bacteria', 'virus'])]
              .assign(y=lambda d: (d.sous_type == 'virus').astype(int)),
    },
}

# Tâches de validation croisée : on entraîne sur un hôpital et on teste sur l'autre
TACHES['tb_shenzhen_vers_montgomery'] = {
    'titre': 'Tuberculose, Shenzhen vers Montgomery', 'libelles': ('Normal', 'Tuberculose'), 'pathologie': True,
    'df': tb.assign(y=(tb.classe == 'TB').astype(int)), 'source': 'Shenzhen', 'cible': 'Montgomery'}
TACHES['tb_montgomery_vers_shenzhen'] = {
    'titre': 'Tuberculose, Montgomery vers Shenzhen', 'libelles': ('Normal', 'Tuberculose'), 'pathologie': True,
    'df': tb.assign(y=(tb.classe == 'TB').astype(int)), 'source': 'Montgomery', 'cible': 'Shenzhen'}

## 3. Partitions
Mêmes partitions par patient que le premier notebook (même graine). Pour la validation croisée,
l'hôpital source est partagé en entraînement (85 %) et validation (15 %), l'hôpital cible sert entièrement de test.

In [ ]:
def partitionner(df, seed=CONFIG['seed']):
    df = df.reset_index(drop=True).copy()
    _, p_val, p_test = CONFIG['partition']
    patients = df.groupby('patient')['y'].agg(lambda s: int(s.mean() >= 0.5)).reset_index()
    reste, test = train_test_split(patients, test_size=p_test, stratify=patients.y, random_state=seed)
    train, val = train_test_split(reste, test_size=p_val / (1 - p_test), stratify=reste.y, random_state=seed)
    ensemble = {**dict.fromkeys(train.patient, 'train'), **dict.fromkeys(val.patient, 'val'),
                **dict.fromkeys(test.patient, 'test')}
    df['split'] = df.patient.map(ensemble)
    assert df.groupby('patient').split.nunique().max() == 1, 'Un patient est dans deux ensembles'
    return df


def partitionner_croise(df, source, cible, seed=CONFIG['seed']):
    src = df[df.source == source].reset_index(drop=True).copy()
    train, val = train_test_split(src.patient.unique(), test_size=0.15, random_state=seed,
                                  stratify=src.groupby('patient').y.first().loc[src.patient.unique()])
    src['split'] = np.where(src.patient.isin(val), 'val', 'train')
    return pd.concat([src, df[df.source == cible].assign(split='test')], ignore_index=True)


DONNEES, lignes_resume = {}, []
for cle, t in TACHES.items():
    d = partitionner_croise(t['df'], t['source'], t['cible']) if 'source' in t else partitionner(t['df'])
    DONNEES[cle] = d
    for split in ('train', 'val', 'test'):
        s = d[d.split == split]
        lignes_resume.append({'tâche': t['titre'], 'ensemble': split, 'images': len(s), 'patients': s.patient.nunique(),
                              'négatifs': int((s.y == 0).sum()), 'positifs': int((s.y == 1).sum())})
REPARTITION = pd.DataFrame(lignes_resume)
REPARTITION.to_csv(OUT_DIR / 'repartition_donnees.csv', index=False)
REPARTITION

In [ ]:
def charger_gris(chemin):
    img = Image.open(chemin)
    if img.mode in ('I;16', 'I;16B', 'I;16L', 'I', 'F'):
        a = np.asarray(img, dtype=np.float32)
        a = (a - a.min()) / max(float(a.max() - a.min()), 1e-6) * 255
        img = Image.fromarray(a.astype(np.uint8))
    return img.convert('L')


def mettre_en_cache(chemin):
    sortie = CACHE_DIR / (hashlib.md5(chemin.encode()).hexdigest() + '.png')
    if not sortie.exists():
        c = CONFIG['taille_cache']
        charger_gris(chemin).resize((c, c), Image.LANCZOS).save(sortie)
    return str(sortie)


tous = sorted(set().union(*(set(d.path) for d in DONNEES.values())))
t0 = time.time()
with ThreadPoolExecutor(8) as pool:
    correspondance = dict(zip(tous, pool.map(mettre_en_cache, tous)))
for d in DONNEES.values():
    d['cache'] = d.path.map(correspondance)
print(f'{len(tous)} images prétraitées en {time.time() - t0:.0f} s')

## 4. Masquage, architectures, modèle

In [ ]:
NORMALISATION = transforms.Normalize([0.485, 0.456, 0.406], [0.229, 0.224, 0.225])
TAILLE = CONFIG['taille_entree']

TF_ENTRAINEMENT = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.RandomAffine(degrees=15, scale=(0.9, 1.1)),
    transforms.RandomResizedCrop(TAILLE, scale=(0.85, 1.0), ratio=(0.95, 1.05)),
    transforms.RandomHorizontalFlip(),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.ToTensor(), NORMALISATION,
])
TF_EVALUATION = transforms.Compose([
    transforms.Grayscale(num_output_channels=3),
    transforms.Resize((TAILLE, TAILLE)),
    transforms.ToTensor(), NORMALISATION,
])


def zone_centrale(h, w, bande=CONFIG['bande_masque']):
    """Masque booléen : True au centre de l'image, False dans la bande de bord."""
    bh, bw = int(round(h * bande)), int(round(w * bande))
    centre = np.zeros((h, w), dtype=bool)
    centre[bh:h - bh, bw:w - bw] = True
    return centre


def appliquer_masque(img, masque):
    """« bords_masques » noircit la bande de bord ; « centre_masque » noircit tout sauf elle.
    Le masque est appliqué avant l'augmentation, sur l'image stockée en 256 × 256."""
    if masque == 'aucun':
        return img
    a = np.array(img)
    centre = zone_centrale(*a.shape)
    a[~centre if masque == 'bords_masques' else centre] = 0
    return Image.fromarray(a)


class Radiographies(Dataset):
    def __init__(self, df, tf, masque):
        self.chemins, self.y, self.tf, self.masque = df.cache.tolist(), df.y.astype('float32').tolist(), tf, masque

    def __len__(self):
        return len(self.chemins)

    def __getitem__(self, i):
        img = appliquer_masque(Image.open(self.chemins[i]).convert('L'), self.masque)
        return self.tf(img), torch.tensor(self.y[i])


def init_worker(worker_id):
    graine = torch.initial_seed() % 2**32
    np.random.seed(graine)
    random.seed(graine)


def chargeurs(df, masque):
    g = torch.Generator().manual_seed(CONFIG['seed'])
    commun = {'batch_size': CONFIG['batch_size'], 'num_workers': CONFIG['num_workers'],
              'pin_memory': DEVICE.type == 'cuda', 'worker_init_fn': init_worker}
    train = df[df.split == 'train']
    return {
        'train': DataLoader(Radiographies(train, TF_ENTRAINEMENT, masque), shuffle=True, generator=g,
                            drop_last=len(train) > CONFIG['batch_size'], **commun),
        'val': DataLoader(Radiographies(df[df.split == 'val'], TF_EVALUATION, masque), shuffle=False, **commun),
        'test': DataLoader(Radiographies(df[df.split == 'test'], TF_EVALUATION, masque), shuffle=False, **commun),
    }


def construire_backbone(archi):
    """Renvoie (couches de convolution, nb de canaux en sortie, ReLU finale ?, calendrier de dégel de la phase 2).
    Le calendrier suit la même logique pour les trois réseaux : dernier étage dès l'époque 1, avant-dernier dès l'époque 6."""
    poids = CONFIG['poids_imagenet']
    if archi == 'densenet121':
        m = torchvision.models.densenet121(weights='IMAGENET1K_V1' if poids else None)
        return m.features, 1024, True, [(0, ['denseblock4', 'norm5']), (5, ['transition3', 'denseblock3'])]
    if archi == 'resnet50':
        m = torchvision.models.resnet50(weights='IMAGENET1K_V1' if poids else None)
        couches = nn.Sequential(OrderedDict(conv1=m.conv1, bn1=m.bn1, relu=m.relu, maxpool=m.maxpool,
                                            layer1=m.layer1, layer2=m.layer2, layer3=m.layer3, layer4=m.layer4))
        return couches, 2048, False, [(0, ['layer4']), (5, ['layer3'])]
    if archi == 'efficientnet_b0':
        m = torchvision.models.efficientnet_b0(weights='IMAGENET1K_V1' if poids else None)
        return m.features, 1280, False, [(0, ['8', '7']), (5, ['6', '5'])]
    raise ValueError(archi)


class CNNRadio(nn.Module):
    """Backbone pré-entraîné, puis GAP → Dropout(0,5) → couche dense à 1 sortie (sigmoïde), comme au Chapitre 2."""

    def __init__(self, archi, dropout=0.5):
        super().__init__()
        self.features, canaux, self.relu_finale, self.degel = construire_backbone(archi)
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(canaux, 1)
        self.activations = self.gradients = None

    def forward(self, x, garder_activations=False):
        a = self.features(x)
        if self.relu_finale:                      # DenseNet ne termine pas par une activation
            a = F.relu(a)
        if garder_activations:
            self.activations = a
            a.register_hook(lambda g: setattr(self, 'gradients', g))
        z = F.adaptive_avg_pool2d(a, 1).flatten(1)
        return self.fc(self.dropout(z)).squeeze(1)


def degeler(model, blocs):
    for nom, module in model.features.named_children():
        if nom in blocs:
            for p in module.parameters():
                p.requires_grad = True


def geler_backbone(model):
    for p in model.features.parameters():
        p.requires_grad = False


def mode_entrainement(model):
    model.train()
    for module in model.features.children():
        if not any(p.requires_grad for p in module.parameters()):
            module.eval()

In [ ]:
def une_epoque(model, loader, critere, optimiseur=None, scaler=None):
    entrainement = optimiseur is not None
    mode_entrainement(model) if entrainement else model.eval()
    perte, correct, n = 0.0, 0, 0
    with torch.set_grad_enabled(entrainement):
        for x, y in loader:
            x, y = x.to(DEVICE, non_blocking=True), y.to(DEVICE, non_blocking=True)
            with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
                logits = model(x)
            loss = critere(logits.float(), y)
            if entrainement:
                optimiseur.zero_grad(set_to_none=True)
                scaler.scale(loss).backward()
                scaler.step(optimiseur)
                scaler.update()
            perte += loss.item() * len(y)
            correct += ((logits > 0).float() == y).sum().item()
            n += len(y)
    return perte / n, correct / n


def entrainer_phase(model, loaders, critere, nom_phase, cfg, historique):
    optimiseur = torch.optim.Adam(model.parameters(), lr=cfg['lr'], weight_decay=CONFIG['weight_decay'])
    planif = torch.optim.lr_scheduler.ReduceLROnPlateau(optimiseur, mode='min', **CONFIG['plateau'])
    scaler = torch.amp.GradScaler(enabled=DEVICE.type == 'cuda')
    meilleure, attente, meilleurs_poids, epoque_retenue = float('inf'), 0, None, None
    for ep in range(cfg['max_epochs']):
        for debut, blocs in cfg.get('degel', []):
            if ep == debut:
                degeler(model, blocs)
                print(f'   dégel : {", ".join(blocs)}')
        t0 = time.time()
        loss, acc = une_epoque(model, loaders['train'], critere, optimiseur, scaler)
        val_loss, val_acc = une_epoque(model, loaders['val'], critere)
        planif.step(val_loss)
        historique.append({'phase': nom_phase, 'epoque': len(historique) + 1, 'loss': loss, 'acc': acc,
                           'val_loss': val_loss, 'val_acc': val_acc,
                           'lr': optimiseur.param_groups[0]['lr'], 'duree_s': time.time() - t0})
        print(f"{nom_phase} | époque {len(historique):>2} | perte {loss:.4f} / val {val_loss:.4f} "
              f"| exactitude {acc:.3f} / val {val_acc:.3f} | {time.time() - t0:.0f} s")
        if val_loss < meilleure - 1e-4:
            meilleure, attente, epoque_retenue = val_loss, 0, len(historique)
            meilleurs_poids = {k: v.detach().cpu().clone() for k, v in model.state_dict().items()}
        else:
            attente += 1
            if attente >= cfg['patience']:
                print(f'   arrêt anticipé (pas d\'amélioration depuis {attente} époques)')
                break
    model.load_state_dict(meilleurs_poids)
    return epoque_retenue


@torch.no_grad()
def predire(model, loader):
    model.eval()
    ys, ps = [], []
    for x, y in loader:
        with torch.autocast(device_type=DEVICE.type, dtype=torch.float16, enabled=DEVICE.type == 'cuda'):
            ps.append(torch.sigmoid(model(x.to(DEVICE)).float()).cpu())
        ys.append(y)
    return torch.cat(ys).numpy().astype(int), torch.cat(ps).numpy()


def metriques(y, p, seuil):
    pred = (p >= seuil).astype(int)
    tn, fp, fn, tp = confusion_matrix(y, pred, labels=[0, 1]).ravel()
    precision = tp / (tp + fp) if tp + fp else 0.0
    rappel = tp / (tp + fn) if tp + fn else 0.0
    return {'precision': precision, 'rappel': rappel,
            'specificite': tn / (tn + fp) if tn + fp else 0.0,
            'f1': 2 * precision * rappel / (precision + rappel) if precision + rappel else 0.0,
            'exactitude': (tp + tn) / len(y),
            'auc': roc_auc_score(y, p) if len(np.unique(y)) == 2 else float('nan'),
            'vp': int(tp), 'fp': int(fp), 'fn': int(fn), 'vn': int(tn)}


def seuil_youden(y, p):
    """Seuil maximisant sensibilité + spécificité − 1, choisi sur la VALIDATION (jamais sur le test)."""
    fpr, tpr, seuils = roc_curve(y, p)
    return float(min(seuils[np.argmax(tpr - fpr)], 1.0))


def intervalles_bootstrap(y, p, seuil, n=CONFIG['bootstrap']):
    rng = np.random.default_rng(CONFIG['seed'])
    tirages = {k: [] for k in ('precision', 'rappel', 'specificite', 'f1', 'auc')}
    for _ in range(n):
        idx = rng.integers(0, len(y), len(y))
        if len(np.unique(y[idx])) < 2:
            continue
        m = metriques(y[idx], p[idx], seuil)
        for k in tirages:
            tirages[k].append(m[k])
    return {k: [float(np.percentile(v, 2.5)), float(np.percentile(v, 97.5))] for k, v in tirages.items()}


def gradcam(model, image_pil):
    model.eval()
    x = TF_EVALUATION(image_pil).unsqueeze(0).to(DEVICE).requires_grad_(True)
    model.zero_grad(set_to_none=True)
    logit = model(x, garder_activations=True)
    logit.sum().backward()                                   # gradient du score « pathologique »
    a, g = model.activations.detach()[0], model.gradients.detach()[0]
    carte = F.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
    carte = carte / (carte.max() + 1e-8)
    carte = F.interpolate(carte[None, None], size=(TAILLE, TAILLE), mode='bilinear', align_corners=False)[0, 0]
    return carte.cpu().numpy(), float(torch.sigmoid(logit).item())


def superposer(image_pil, carte, alpha=0.4):
    fond = np.asarray(image_pil.convert('L').resize((TAILLE, TAILLE)), dtype=np.float32)[..., None] / 255
    chaleur = matplotlib.colormaps['jet'](carte)[..., :3]
    return np.clip((1 - alpha) * fond + alpha * chaleur, 0, 1)


def sauver_gradcam(model, test, dossier):
    """Un VP représentatif (score médian parmi les VP) et le FN le plus franc, plus une galerie pour choisir."""
    test = test.reset_index(drop=True)
    vp = test[(test.y == 1) & (test.pred == 1)].sort_values('proba')
    fn = test[(test.y == 1) & (test.pred == 0)].sort_values('proba')
    choix = {}
    if len(vp):
        choix['VP'] = vp.iloc[len(vp) // 2]
    if len(fn):
        choix['FN'] = fn.iloc[0]
    galerie = list(vp.iloc[np.linspace(0, len(vp) - 1, min(len(vp), 6)).astype(int)].itertuples()) if len(vp) else []
    galerie += list(fn.head(6).itertuples())
    (dossier / 'gradcam_galerie').mkdir(exist_ok=True)
    infos = {}
    for type_cas, ligne in choix.items():
        img = Image.open(ligne.cache)
        carte, proba = gradcam(model, img)
        plt.imsave(dossier / f'gradcam_{type_cas}.png', superposer(img, carte))
        infos[type_cas] = {'image': ligne.path, 'patient': ligne.patient, 'proba': proba}
    for r in galerie:
        img = Image.open(r.cache)
        carte, _ = gradcam(model, img)
        type_cas = 'VP' if r.pred == 1 else 'FN'
        plt.imsave(dossier / 'gradcam_galerie' / f'{type_cas}_{r.proba:.3f}_{r.patient}.png', superposer(img, carte))
    return infos


def gradcam_masque(model, image_pil):
    """Grad-CAM sur une image déjà masquée ; renvoie la carte en 224 × 224 et la probabilité."""
    model.eval()
    x = TF_EVALUATION(image_pil).unsqueeze(0).to(DEVICE).requires_grad_(True)
    model.zero_grad(set_to_none=True)
    logit = model(x, garder_activations=True)
    logit.sum().backward()
    a, g = model.activations.detach()[0], model.gradients.detach()[0]
    carte = F.relu((g.mean(dim=(1, 2))[:, None, None] * a).sum(0))
    carte = carte / (carte.max() + 1e-8)
    carte = F.interpolate(carte[None, None], size=(TAILLE, TAILLE), mode='bilinear', align_corners=False)[0, 0]
    return carte.cpu().numpy(), float(torch.sigmoid(logit).item())


BORDS_224 = ~zone_centrale(TAILLE, TAILLE)


def part_activation_bords(model, test, masque):
    """Part moyenne de l'activation Grad-CAM située dans la bande de bord, sur les cas positifs du test.
    Repère : une activation uniforme donnerait la part de surface de la bande."""
    parts = []
    for chemin in test[test.y == 1].cache:
        carte, _ = gradcam_masque(model, appliquer_masque(Image.open(chemin).convert('L'), masque))
        total = carte.sum()
        if total > 0:
            parts.append(float(carte[BORDS_224].sum() / total))
    return float(np.mean(parts)) if parts else float('nan'), float(BORDS_224.mean())

## 5. Lancement des expériences

In [ ]:
RESULTATS = {}
for tache, archi, masque in EXPERIENCES:
    nom = nom_experience(tache, archi, masque)
    dossier = OUT_DIR / nom
    dossier.mkdir(exist_ok=True)
    if (dossier / 'metriques.json').exists() and not REENTRAINER:
        RESULTATS[nom] = json.loads((dossier / 'metriques.json').read_text(encoding='utf-8'))
        print(f'✔ {nom} déjà terminé, sauté')
        continue

    print(f'\n===== {nom} =====')
    df = DONNEES[tache]
    fixer_graine()
    loaders = chargeurs(df, masque)
    model = CNNRadio(archi, CONFIG['dropout']).to(DEVICE)
    train = df[df.split == 'train']
    poids_positif = (train.y == 0).sum() / max((train.y == 1).sum(), 1)
    critere = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(poids_positif), device=DEVICE))

    historique, debut = [], time.time()
    geler_backbone(model)
    entrainer_phase(model, loaders, critere, 'Phase 1', CONFIG['phase1'], historique)
    fin_phase1 = len(historique)
    epoque_retenue = entrainer_phase(model, loaders, critere, 'Phase 2', {**CONFIG['phase2'], 'degel': model.degel},
                                     historique)
    duree = time.time() - debut
    pd.DataFrame(historique).to_csv(dossier / 'historique.csv', index=False)

    y_val, p_val = predire(model, loaders['val'])
    seuil = seuil_youden(y_val, p_val)
    y_test, p_test = predire(model, loaders['test'])
    test = df[df.split == 'test'].reset_index(drop=True).assign(proba=p_test, pred=(p_test >= seuil).astype(int))
    test[['path', 'source', 'patient', 'y', 'proba', 'pred']].to_csv(dossier / 'predictions_test.csv', index=False)
    part_bords, surface_bords = part_activation_bords(model, test, masque)

    RESULTATS[nom] = {
        'tache': tache, 'architecture': archi, 'masque': masque, 'titre': TACHES[tache]['titre'],
        'seuil_validation': seuil,
        'test': metriques(y_test, p_test, seuil),
        'ic95': intervalles_bootstrap(y_test, p_test, seuil),
        'n_test': len(y_test), 'n_test_positifs': int(y_test.sum()),
        'epoques_phase1': fin_phase1, 'epoques_total': len(historique), 'epoque_retenue': epoque_retenue,
        'duree_entrainement_min': duree / 60,
        'part_activation_bords': part_bords, 'surface_bords': surface_bords,
        'parametres_millions': sum(p.numel() for p in model.parameters()) / 1e6,
    }
    (dossier / 'metriques.json').write_text(json.dumps(RESULTATS[nom], indent=2, ensure_ascii=False), encoding='utf-8')
    torch.save(model.state_dict(), dossier / 'modele.pt')
    m = RESULTATS[nom]['test']
    print(f"→ test : AUC {m['auc']:.3f} | rappel {m['rappel']:.3f} | spécificité {m['specificite']:.3f} "
          f"| activation sur les bords {part_bords:.0%} (surface {surface_bords:.0%}) | {duree / 60:.0f} min")
    del model
    if DEVICE.type == 'cuda':
        torch.cuda.empty_cache()

## 6. Synthèse
Les écarts d'AUC sont estimés par bootstrap **apparié** : les mêmes patients sont rééchantillonnés pour les deux
modèles comparés, ce qui tient compte du fait qu'ils sont évalués sur le même ensemble de test.

In [ ]:
def fr(x, d=3):
    return f'{x:.{d}f}'.replace('.', ',')


def ecart_auc_apparie(nom_a, nom_b, n=CONFIG['bootstrap']):
    """AUC(b) − AUC(a) et son IC 95 % par bootstrap apparié sur les patients du test commun."""
    a = pd.read_csv(OUT_DIR / nom_a / 'predictions_test.csv')
    b = pd.read_csv(OUT_DIR / nom_b / 'predictions_test.csv')
    m = a.merge(b[['path', 'proba']], on='path', suffixes=('_a', '_b'))
    y, pa, pb = m.y.values, m.proba_a.values, m.proba_b.values
    rng = np.random.default_rng(CONFIG['seed'])
    ecarts = []
    for _ in range(n):
        i = rng.integers(0, len(y), len(y))
        if len(np.unique(y[i])) == 2:
            ecarts.append(roc_auc_score(y[i], pb[i]) - roc_auc_score(y[i], pa[i]))
    return roc_auc_score(y, pb) - roc_auc_score(y, pa), np.percentile(ecarts, 2.5), np.percentile(ecarts, 97.5)


lignes = []
for tache, archi, masque in EXPERIENCES:
    nom = nom_experience(tache, archi, masque)
    if nom not in RESULTATS:
        continue
    r, m = RESULTATS[nom], RESULTATS[nom]['test']
    ligne = {'Tâche': r['titre'], 'Architecture': archi, 'Masque': masque,
             'AUC [IC 95 %]': f"{fr(m['auc'])} [{fr(r['ic95']['auc'][0])} ; {fr(r['ic95']['auc'][1])}]",
             'Rappel': fr(m['rappel']), 'Spécificité': fr(m['specificite']), 'F1': fr(m['f1']),
             'Activation sur les bords': f"{r['part_activation_bords']:.0%} (surface {r['surface_bords']:.0%})",
             'Paramètres (M)': fr(r['parametres_millions'], 1), 'Durée (min)': round(r['duree_entrainement_min'])}
    reference = nom_experience(tache, 'densenet121', 'aucun')
    if nom != reference and reference in RESULTATS and tache in ('pneumonie', 'tuberculose', 'bacterien_viral'):
        d, bas, haut = ecart_auc_apparie(reference, nom)
        ligne['Écart d\'AUC vs DenseNet complet [IC 95 %]'] = f"{d:+.3f} [{bas:+.3f} ; {haut:+.3f}]".replace('.', ',')
    lignes.append(ligne)
SYNTHESE = pd.DataFrame(lignes).fillna('')
SYNTHESE.to_csv(OUT_DIR / 'synthese_complements.csv', index=False, encoding='utf-8-sig')
SYNTHESE

## 7. Figure du test de raccourci

In [ ]:
# Même patient de test, trois versions de l'image et la carte Grad-CAM de chaque modèle
noms = [nom_experience('pneumonie', 'densenet121', m) for m in ('aucun', 'bords_masques', 'centre_masque')]
if all((OUT_DIR / n / 'modele.pt').exists() for n in noms):
    test = DONNEES['pneumonie'][DONNEES['pneumonie'].split == 'test']
    positifs = test[test.y == 1]
    exemple = positifs[positifs.patient == 'P-person402'].iloc[0] if (positifs.patient == 'P-person402').any() else positifs.iloc[0]
    titres = {'aucun': 'Image complète', 'bords_masques': 'Bords masqués', 'centre_masque': 'Bords seuls'}
    fig, axes = plt.subplots(2, 3, figsize=(10, 7))
    for j, (nom, masque) in enumerate(zip(noms, ('aucun', 'bords_masques', 'centre_masque'))):
        model = CNNRadio('densenet121').to(DEVICE)
        model.load_state_dict(torch.load(OUT_DIR / nom / 'modele.pt', map_location=DEVICE))
        img = appliquer_masque(Image.open(exemple.cache).convert('L'), masque)
        carte, proba = gradcam_masque(model, img)
        fond = np.asarray(img.resize((TAILLE, TAILLE)), dtype=np.float32) / 255
        auc = RESULTATS[nom]['test']['auc']
        axes[0][j].imshow(fond, cmap='gray')
        axes[0][j].set_title(f"{titres[masque]}\nAUC de test = {fr(auc)}", fontsize=10)
        axes[1][j].imshow(np.clip(0.6 * fond[..., None] + 0.4 * matplotlib.colormaps['jet'](carte)[..., :3], 0, 1))
        axes[1][j].set_title(f'Grad-CAM, score = {fr(proba, 2)}', fontsize=10)
        for ax in (axes[0][j], axes[1][j]):
            ax.axis('off')
        del model
    plt.tight_layout()
    plt.savefig(OUT_DIR / 'figure_test_raccourci.png', bbox_inches='tight')
    plt.show()

## 8. Archive

In [ ]:
texte = ['# Expériences complémentaires : récapitulatif', '',
         f"Exécuté le {datetime.date.today().isoformat()} · {GPU} · PyTorch {torch.__version__} · "
         f"torchvision {torchvision.__version__} · Python {platform.python_version()}", '',
         '## Configuration', '', '```json', json.dumps(CONFIG, indent=2, ensure_ascii=False), '```', '',
         '## Répartition', '', REPARTITION.to_markdown(index=False), '',
         '## Synthèse', '', SYNTHESE.to_markdown(index=False), '']
(OUT_DIR / 'resume_complements.md').write_text('\n'.join(texte), encoding='utf-8')
print('\n'.join(texte))

archive = Path('/content/memoire_complements.zip' if IN_COLAB else 'memoire_complements.zip')
with zipfile.ZipFile(archive, 'w', zipfile.ZIP_DEFLATED) as z:
    for f in OUT_DIR.rglob('*'):
        if f.is_file() and f.suffix != '.pt':
            z.write(f, f.relative_to(OUT_DIR))
if IN_COLAB:
    colab_files.download(archive)
print(f'Archive : {archive.resolve()}')